# 02_exploratory_analysis

## 目的
- 価格分布・都道府県別件数・市区町村別平均
- 上昇率ランキング・地図表示
- ノートブックから DB を直接使う探索分析のデモ

In [15]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# プロジェクトルートを config.py の存在で探索（CWD に依存しない）
def _find_project_root() -> str:
    try:
        _nb_dir = Path(globals()["_dh"][0])
    except (KeyError, IndexError):
        _nb_dir = Path.cwd()
    for candidate in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]:
        if (candidate / "config.py").exists():
            return str(candidate.resolve())
    fallback = Path("/home/kazumasa/projects/land_price_api_app")
    if fallback.exists():
        return str(fallback)
    raise FileNotFoundError("プロジェクトルートが見つかりません")

_project_root = _find_project_root()
os.chdir(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)
print("作業ディレクトリ:", os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
作業ディレクトリ: /home/kazumasa/projects/land_price_api_app


## 1. データ読み込み

In [16]:
from notebook_utils import load_env_and_connect, load_year_from_db, show_basic_summary
import db
from config import PROCESSED_DIR

conn = load_env_and_connect(read_only=True)

if conn is not None:
    years = db.get_available_years(conn)
else:
    pqs = sorted(PROCESSED_DIR.glob("land_prices_y*_pc0.parquet"))
    years = sorted(set(int(p.stem.split("_y")[1].split("_")[0]) for p in pqs), reverse=True)

if not years:
    print("データがありません。01_sync_and_normalize.ipynb を先に実行してください。")
else:
    YEAR = years[0]  # 最新年
    df = load_year_from_db(conn, YEAR)
    show_basic_summary(df)

✓ APIキーを確認しました
✓ DB 接続成功 (読み取り専用): 8,188 レコード / 年度: [2026, 2025]
✓ DB から 4,094 件ロード (year=2026)
総件数         : 4,094
年度           : [2026]
都道府県数     : 5
市区町村数     : 107
平均価格       : ¥1,061,890/m²
中央値価格     : ¥378,000/m²
最高価格       : ¥67,100,000/m²
平均前年比     : +6.95%


## 2. 価格分布

In [17]:
import plotly.express as px
import pandas as pd

if "df" in dir() and not df.empty:
    fig = px.histogram(
        df.dropna(subset=["price_yen_per_sqm"]),
        x="price_yen_per_sqm",
        nbins=50,
        log_x=True,
        title=f"公示価格分布 ({YEAR}年) - 対数スケール",
        labels={"price_yen_per_sqm": "価格 (円/m²)"},
        color="use_category_name",
    )
    fig.show()

## 3. 都道府県別件数

In [18]:
from analytics import compute_pref_summary

if "df" in dir() and not df.empty:
    pref_df = compute_pref_summary(df)
    fig = px.bar(
        pref_df.sort_values("point_count", ascending=False).head(30),
        x="prefecture_name", y="point_count",
        title="都道府県別地点数",
        labels={"prefecture_name": "都道府県", "point_count": "地点数"},
        color="avg_price", color_continuous_scale="Blues",
    )
    fig.show()
    display(pref_df.head(20))

,year,prefecture_code,prefecture_name,point_count,avg_price,median_price,avg_yoy_pct
2,2026,13,東京都,2542,1.490482e+06,564000.0,8.467807
3,2026,14,神奈川県,625,3.891112e+05,302000.0,4.777905
1,2026,12,千葉県,129,3.623240e+05,276000.0,7.244444
0,2026,11,埼玉県,720,2.777089e+05,201000.0,3.582535
4,2026,19,山梨県,14,5.193571e+04,51300.0,-0.278571


## 4. 市区町村別平均価格（上位20）

In [19]:
from notebook_utils import plot_city_ranking

if "df" in dir() and not df.empty:
    plot_city_ranking(df, year=YEAR, top_n=20)

## 5. 上昇率ランキング

In [20]:
from analytics import compute_yoy_rankings

if "df" in dir() and not df.empty:
    up_df = compute_yoy_rankings(df, top_n=20, ascending=False, year=YEAR)
    if not up_df.empty:
        display(up_df[["location_text", "city_name", "price_yen_per_sqm", "yoy_change_pct"]])
    else:
        print("前年比データがありません（単年同期の場合は取得できません）")

,location_text,city_name,price_yen_per_sqm,yoy_change_pct
0,東京都渋谷区桜丘町１５番６外,渋谷区,4450000.0,29.0
1,東京都台東区浅草１丁目１６番１４外,台東区,9150000.0,27.6
2,東京都台東区西浅草２丁目６６番２,台東区,3230000.0,25.2
3,東京都台東区浅草２丁目２４番６,台東区,1240000.0,24.2
4,東京都渋谷区渋谷２丁目６番９外,渋谷区,3600000.0,24.1
5,東京都中野区中野３丁目１０７番１０外,中野区,8270000.0,24.0
6,東京都台東区浅草２丁目５２番１１,台東区,2490000.0,23.9
7,東京都台東区浅草５丁目７７番１５,台東区,988000.0,23.8
8,東京都台東区西浅草３丁目２番１０,台東区,1610000.0,23.8
9,東京都中野区中野５丁目２１番７,中野区,5410000.0,23.8


## 6. 地図表示

In [21]:
from notebook_utils import plot_points_map

if "df" in dir() and not df.empty:
    plot_points_map(df, title=f"公示地価マップ ({YEAR}年)")

## 7. 沖縄県 上昇率ランキング

In [23]:
# 沖縄県 上昇率ランキング
import db as _db
import pandas as pd
from config import PROCESSED_DIR
from analytics import compute_yoy_rankings
import plotly.express as px

# DBから取得 → なければ全Parquetから絞り込む
df_okinawa_all = pd.DataFrame()
if conn is not None:
    df_okinawa_all = _db.read_land_prices(conn, filters={"prefecture_code": "47"})

if df_okinawa_all.empty:
    _pqs = sorted(PROCESSED_DIR.glob("land_prices_y*_pc0.parquet"))
    _dfs = []
    for p in _pqs:
        _tmp = pd.read_parquet(p)
        _filtered = _tmp[_tmp["prefecture_code"] == "47"]
        if not _filtered.empty:
            _dfs.append(_filtered)
    df_okinawa_all = pd.concat(_dfs, ignore_index=True) if _dfs else pd.DataFrame()
    if not df_okinawa_all.empty:
        print(f"✓ Parquet から沖縄データ {len(df_okinawa_all):,} 件ロード")

if df_okinawa_all.empty:
    print("沖縄データがありません。01_sync_and_normalize.ipynb で PREF_CODE='47' を実行してください。")
else:
    OKI_YEAR = df_okinawa_all["year"].max()
    print(f"沖縄データ: {len(df_okinawa_all):,} 件 / 最新年: {OKI_YEAR}")
    oki_up = compute_yoy_rankings(df_okinawa_all, top_n=30, ascending=False, year=OKI_YEAR)

    if oki_up.empty:
        print("前年比データがありません（yoy_change_pct がすべて NaN）")
    else:
        display(oki_up[["location_text", "city_name", "use_category_name",
                         "price_yen_per_sqm", "yoy_change_pct"]].head(20))

        fig = px.bar(
            oki_up.head(20).sort_values("yoy_change_pct"),
            x="yoy_change_pct", y="location_text",
            orientation="h",
            color="use_category_name",
            title=f"沖縄県 上昇率 TOP20 ({OKI_YEAR}年)",
            labels={"yoy_change_pct": "前年比 (%)", "location_text": "地点"},
        )
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
        fig.show()

✓ Parquet から沖縄データ 384 件ロード
沖縄データ: 384 件 / 最新年: 2026


,location_text,city_name,use_category_name,price_yen_per_sqm,yoy_change_pct
0,沖縄県国頭郡本部町字大浜大崎原８７９番４,本部町,商業地,66800.0,22.1
1,沖縄県宜野湾市新城２丁目４３６番３,宜野湾市,商業地,185000.0,19.4
2,沖縄県宜野湾市野嵩３丁目１３０５番１,宜野湾市,住宅地,121000.0,18.6
3,沖縄県宮古島市上野字野原東方原１１０４番,宮古島市,住宅地,15500.0,16.5
4,沖縄県宮古島市平良字西里羽立３９１番外,宮古島市,商業地,149000.0,14.6
5,沖縄県宜野湾市普天間２丁目４８６番３外,宜野湾市,商業地,128000.0,14.3
6,沖縄県宮古島市平良字下里西里６１番外,宮古島市,商業地,73500.0,12.7
7,沖縄県うるま市石川東恩納前原７２２番１０,うるま市,住宅地,53500.0,12.6
8,沖縄県宮古島市平良字西里前比屋２７３番,宮古島市,住宅地,64800.0,12.3
9,沖縄県中頭郡北中城村字喜舎場東前原３６７番１０,北中城村,住宅地,91800.0,12.1


## 8. 特定地点の過去10年価格推移

In [24]:
import time
import pandas as pd
import plotly.graph_objects as go
import api_client
import normalize
from tiles import lonlat_to_tile_indices

# ---- 対象地点の検索キーワード ----
SEARCH_TEXT = "大崎原"  # ← 変更可能（部分一致）

# 既取得データから地点を特定
match = df_okinawa_all[df_okinawa_all["location_text"].str.contains(SEARCH_TEXT, na=False)]
if match.empty:
    print(f"'{SEARCH_TEXT}' に一致する地点が見つかりません。")
    print("以下から手動で確認してください:")
    display(df_okinawa_all[["point_id", "location_text", "city_name"]].head(20))
else:
    display(match[["point_id", "location_text", "city_name", "lon", "lat",
                   "price_yen_per_sqm", "yoy_change_pct", "year"]])


,point_id,location_text,city_name,lon,lat,price_yen_per_sqm,yoy_change_pct,year
184,12025950,沖縄県国頭郡本部町字大浜大崎原８７９番４,本部町,127.884668,26.656683,54700.0,8.5,2025
376,12025950,沖縄県国頭郡本部町字大浜大崎原８７９番４,本部町,127.884668,26.656683,66800.0,22.1,2026


In [25]:
# ---- 地点を確定して過去10年分を取得 ----
# 上のセルで目的の地点が表示されたら、point_id を確認してここを実行

if match.empty:
    print("先に上のセルで地点を確認してください。")
else:
    base = match.iloc[0]
    TARGET_POINT_ID = base["point_id"]
    lon, lat = float(base["lon"]), float(base["lat"])
    location_label = base["location_text"]
    print(f"対象地点: {location_label}")
    print(f"point_id: {TARGET_POINT_ID}  /  lon={lon:.6f}, lat={lat:.6f}")

    Z = 13
    FETCH_YEARS = list(range(2016, 2027))  # 過去10年 + 最新

    # --- 既存データから取得済み年をスキップ ---
    existing_years = set(df_okinawa_all[df_okinawa_all["point_id"] == TARGET_POINT_ID]["year"].tolist())
    need_years = [y for y in FETCH_YEARS if y not in existing_years]
    print(f"既取得年: {sorted(existing_years)}  /  APIから取得する年: {need_years}")

    # --- タイル座標計算（中心タイル ± 周囲1タイル計9タイル） ---
    cx, cy = lonlat_to_tile_indices(lon, lat, Z)
    neighbor_tiles = [
        (cx + dx, cy + dy)
        for dx in (-1, 0, 1)
        for dy in (-1, 0, 1)
    ]

    # --- 年ごとに対象タイルだけ取得 ---
    fetched_dfs = []
    for year in need_years:
        yr_features = []
        for tx, ty in neighbor_tiles:
            try:
                data = api_client.fetch_geojson_tile_xpt002(Z, tx, ty, year=year)
                yr_features.extend(data.get("features", []))
                time.sleep(0.2)
            except Exception as e:
                print(f"  {year} タイル({tx},{ty}) スキップ: {e}")
        if yr_features:
            df_yr = normalize.normalize_features_to_dataframe(yr_features, year, 0)
            hit = df_yr[df_yr["point_id"] == TARGET_POINT_ID]
            if not hit.empty:
                fetched_dfs.append(hit)
                print(f"  {year}年: ✓ 取得")
            else:
                print(f"  {year}年: 地点なし（年度未公開の可能性）")
        else:
            print(f"  {year}年: データなし")

    # --- 既存データ + 新規取得を結合 ---
    existing_rows = df_okinawa_all[df_okinawa_all["point_id"] == TARGET_POINT_ID]
    all_parts = [existing_rows] + fetched_dfs
    df_history = pd.concat(all_parts, ignore_index=True).sort_values("year")
    print(f"\n取得完了: {len(df_history)} 年分")
    display(df_history[["year", "price_yen_per_sqm", "yoy_change_pct", "location_text"]])


対象地点: 沖縄県国頭郡本部町字大浜大崎原８７９番４
point_id: 12025950  /  lon=127.884668, lat=26.656683
既取得年: [2025, 2026]  /  APIから取得する年: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
  2016年: ✓ 取得
  2017年: ✓ 取得
  2018年: ✓ 取得
  2019年: ✓ 取得
  2020年: ✓ 取得
  2021年: ✓ 取得
  2022年: ✓ 取得
  2023年: ✓ 取得
  2024年: ✓ 取得

取得完了: 11 年分


,year,price_yen_per_sqm,yoy_change_pct,location_text
2,2016,43200.0,-0.2,沖縄県国頭郡本部町字大浜大崎原８７９番４
3,2017,43100.0,-0.2,沖縄県国頭郡本部町字大浜大崎原８７９番４
4,2018,43100.0,0.0,沖縄県国頭郡本部町字大浜大崎原８７９番４
5,2019,43200.0,0.2,沖縄県国頭郡本部町字大浜大崎原８７９番４
6,2020,46200.0,6.9,沖縄県国頭郡本部町字大浜大崎原８７９番４
7,2021,46100.0,-0.2,沖縄県国頭郡本部町字大浜大崎原８７９番４
8,2022,46100.0,0.0,沖縄県国頭郡本部町字大浜大崎原８７９番４
9,2023,46600.0,1.1,沖縄県国頭郡本部町字大浜大崎原８７９番４
10,2024,50400.0,8.2,沖縄県国頭郡本部町字大浜大崎原８７９番４
0,2025,54700.0,8.5,沖縄県国頭郡本部町字大浜大崎原８７９番４


In [26]:
# ---- グラフ表示 ----
if "df_history" in dir() and not df_history.empty:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_history["year"],
        y=df_history["price_yen_per_sqm"],
        mode="lines+markers",
        name="価格 (円/m²)",
        line=dict(width=2, color="#2196F3"),
        marker=dict(size=8),
    ))

    yoy = df_history["yoy_change_pct"].fillna(0)
    fig.add_trace(go.Bar(
        x=df_history["year"],
        y=yoy,
        name="前年比 (%)",
        yaxis="y2",
        marker_color=["#e53935" if v > 0 else "#1e88e5" for v in yoy],
        opacity=0.6,
    ))

    fig.update_layout(
        title=f"年次推移: {location_label}",
        xaxis=dict(title="年", dtick=1),
        yaxis=dict(title="価格 (円/m²)", tickformat=",.0f"),
        yaxis2=dict(title="前年比 (%)", overlaying="y", side="right"),
        legend=dict(x=0.01, y=0.99),
        height=450,
    )
    fig.show()
